# Lesson 11: Text as tensors

Goes with the Lesson 11 page of the Forkcast course. Tokens, a vocabulary, padding and masks, embeddings for words, and a bag-of-embeddings sentiment model, first on the bundled templated reviews, then on real Yelp reviews.

Runtime: CPU is fine for the templated corpus; a GPU makes the Yelp run quick.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/text.py`

Write `tokenize`, `Vocab`, `pad_batch`, `collate_reviews` and `BagOfEmbeddings` as the page builds them; the dataset and loader helpers are provided.

In [ ]:
%%writefile forkcast/text.py
"""Lesson 11. Text as tensors: tokens, a vocabulary, padding and masks, and a bag-of-embeddings classifier.

``load_reviews`` and ``make_text_loaders`` are provided; the rest are exercises.
"""

from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

PAD, UNK = "<pad>", "<unk>"
SAMPLE = Path(__file__).resolve().parent.parent / "data" / "reviews_mini.csv"
_TOKEN = re.compile(r"[a-z0-9']+|[!?.,]")


def tokenize(text: str) -> list[str]:
    """Lower-case the text and split it into words and the punctuation marks ! ? . ,

    "Great food, FAST delivery!" -> ["great", "food", ",", "fast", "delivery", "!"]
    Use ``_TOKEN.findall`` on the lower-cased text.
    """
    raise NotImplementedError("Lesson 11, step 2: _TOKEN.findall(text.lower())")


class Vocab:
    """Token <-> integer id. Index 0 is PAD, index 1 is UNK, then tokens by decreasing frequency.

    ``min_freq`` drops rare tokens (they become UNK); ``max_size`` caps the total size including the two specials.
    Build it from the training texts only.
    """

    def __init__(self, token_lists, min_freq: int = 1, max_size: int | None = None):
        raise NotImplementedError("Lesson 11, step 3: count tokens with Counter; keep those with count >= min_freq, most common first; self.itos = [PAD, UNK] + kept[:max_size - 2]; self.stoi = {token: i}")

    def __len__(self) -> int:
        return len(self.itos)

    def encode(self, tokens: list[str]) -> list[int]:
        """Tokens to ids; unknown tokens become the UNK id (1)."""
        raise NotImplementedError("Lesson 11, step 3: [self.stoi.get(t, 1) for t in tokens]")

    def decode(self, ids) -> list[str]:
        return [self.itos[int(i)] for i in ids]


def pad_batch(sequences: list[Tensor], pad_idx: int = 0) -> tuple[Tensor, Tensor]:
    """A list of 1-d int64 tensors of different lengths -> (ids (N, T), mask (N, T) bool).

    T is the longest length in the batch. ``pad_sequence(..., batch_first=True, padding_value=pad_idx)``
    does the padding; the mask is True where there is a real token.
    """
    raise NotImplementedError("Lesson 11, step 4: ids = pad_sequence(...); mask = ids != pad_idx; return ids, mask")


class ReviewsDataset(Dataset):
    """One review per item: (ids (T,) int64, label () int64), truncated to ``max_len`` tokens."""

    def __init__(self, texts, labels, vocab: Vocab, max_len: int = 200):
        self.ids = [torch.tensor(vocab.encode(tokenize(t))[:max_len], dtype=torch.long) for t in texts]
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int):
        return self.ids[i], self.labels[i]


def collate_reviews(batch) -> tuple[Tensor, Tensor, Tensor]:
    """The DataLoader's collate_fn: a list of (ids, label) -> (ids (N, T), mask (N, T), labels (N,)).

    Reviews have different lengths, so the default stacking fails; pad them with ``pad_batch``.
    The toolkit expects the target last, so return ids and mask first.
    """
    raise NotImplementedError("Lesson 11, step 4: ids, labels = zip(*batch); padded, mask = pad_batch(list(ids)); return padded, mask, torch.stack(labels)")


class BagOfEmbeddings(nn.Module):
    """Embed every token, average the real ones, classify the average.

    (N, T) ids -> Embedding(vocab, dim, padding_idx=0) -> (N, T, dim) -> masked mean over T -> (N, dim) -> MLP -> (N, n_classes)
    The masked mean multiplies by ``mask[..., None]``, sums over T, and divides by ``mask.sum(1, keepdim=True).clamp_min(1)``.
    """

    def __init__(self, vocab_size: int, dim: int = 64, hidden: int = 64, n_classes: int = 2):
        super().__init__()
        raise NotImplementedError("Lesson 11, step 5: self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0); self.head = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))")

    def forward(self, ids: Tensor, mask: Tensor) -> Tensor:
        raise NotImplementedError("Lesson 11, step 5: emb = self.embedding(ids); masked mean over dim 1; return self.head(pooled)")


def load_reviews(path: str | Path = SAMPLE) -> pd.DataFrame:
    """Columns: text, label (1 = positive, 0 = negative)."""
    return pd.read_csv(path)


def make_text_loaders(train_texts, train_labels, val_texts, val_labels, vocab: Vocab, batch_size: int = 64, max_len: int = 200) -> tuple[DataLoader, DataLoader]:
    """Loaders over ReviewsDatasets with collate_reviews; the vocabulary comes from the training texts."""
    train_ds = ReviewsDataset(train_texts, train_labels, vocab, max_len)
    val_ds = ReviewsDataset(val_texts, val_labels, vocab, max_len)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_reviews),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_reviews))

## 3. Tokens and ids

In [ ]:
import importlib, torch
from torch import nn
import matplotlib.pyplot as plt
from forkcast import text, train
from forkcast.model import count_parameters
importlib.reload(text)

frame = text.load_reviews()
print(frame.head(3).to_string())
print(text.tokenize(frame["text"][0]))
cut = int(0.8 * len(frame)); tr, va = frame.iloc[:cut], frame.iloc[cut:]
vocab = text.Vocab([text.tokenize(t) for t in tr["text"]], min_freq=1)        # training texts only
print(len(vocab), vocab.itos[:8])
ids = vocab.encode(text.tokenize("The pad thai was delicious and the delivery was late. Mixed feelings!"))
print(ids); print(vocab.decode(ids))

## 4. Padding, masks, loaders

In [ ]:
importlib.reload(text)
train_loader, val_loader = text.make_text_loaders(tr["text"], tr["label"], va["text"], va["label"], vocab, batch_size=32)
ids, mask, labels = next(iter(train_loader))
print(ids.shape, mask.shape, labels.shape, ids.dtype, mask.dtype)
print("real lengths:", mask.sum(dim=1)[:8].tolist())
print(ids[0]); print(mask[0].int())

## 5. The model, and the mask at work

In [ ]:
importlib.reload(text)
model = text.BagOfEmbeddings(len(vocab), dim=16, hidden=16)
short = torch.tensor([[5, 6, 7]]); padded = torch.tensor([[5, 6, 7, 0, 0, 0]])
print("padding changes nothing:", torch.allclose(model(short, short != 0), model(padded, padded != 0)))
print("pad row:", model.embedding.weight[0])

torch.manual_seed(0)
model = text.BagOfEmbeddings(len(vocab), dim=32, hidden=32)
print(count_parameters(model), "parameters")
h = train.fit_v2(model, train_loader, val_loader, epochs=8, lr=3e-3, loss_fn=nn.CrossEntropyLoss())
print([round(v, 3) for v in h["val"]])

def accuracy_text(m, loader, device="cpu"):
    m.eval(); correct = total = 0
    with torch.no_grad():
        for i_, mk, lb in loader:
            correct += (m(i_.to(device), mk.to(device)).argmax(1).cpu() == lb).sum().item(); total += len(lb)
    return correct / total
print("validation accuracy on templated reviews:", accuracy_text(model, val_loader))

## 6. Real reviews: Yelp polarity

40,000 reviews streamed from the Hugging Face Hub; vocabulary of the 20,000 most frequent tokens from the training part; reviews truncated to 200 tokens.

In [ ]:
!pip install -q datasets
from datasets import load_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stream = load_dataset("fancyzhx/yelp_polarity", split="train", streaming=True).shuffle(seed=0, buffer_size=10_000)
texts, labels = [], []
for ex in stream:
    texts.append(ex["text"]); labels.append(ex["label"])
    if len(texts) >= 40_000:
        break
n_val = 4_000
yelp_vocab = text.Vocab([text.tokenize(t) for t in texts[n_val:]], min_freq=2, max_size=20_000)
print("vocabulary:", len(yelp_vocab))
y_train, y_val = text.make_text_loaders(texts[n_val:], labels[n_val:], texts[:n_val], labels[:n_val], yelp_vocab, batch_size=128, max_len=200)
torch.manual_seed(0)
yelp_model = text.BagOfEmbeddings(len(yelp_vocab), dim=64, hidden=64)
h = train.fit_v2(yelp_model, y_train, y_val, epochs=5, lr=3e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
print("Yelp validation accuracy:", round(accuracy_text(yelp_model, y_val, device), 3))

emb = yelp_model.embedding.weight.detach().cpu(); normed = emb / emb.norm(dim=1, keepdim=True).clamp_min(1e-8)
for w in ["delicious", "rude", "overpriced", "friendly"]:
    if w in yelp_vocab.stoi:
        sims = normed @ normed[yelp_vocab.stoi[w]]
        print(f"nearest to {w!r}:", [yelp_vocab.itos[j] for j in sims.argsort(descending=True)[1:7].tolist()])

### Break it on purpose: drop the mask

A plain mean over T divides by the padded length. Short reviews suffer most.

In [ ]:
class Unmasked(text.BagOfEmbeddings):
    def forward(self, ids, mask):
        return self.head(self.embedding(ids).mean(dim=1))       # zeros included

torch.manual_seed(0)
bad = Unmasked(len(yelp_vocab), dim=64, hidden=64)
train.fit_v2(bad, y_train, y_val, epochs=5, lr=3e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
print("without the mask:", round(accuracy_text(bad, y_val, device), 3), "| with:", round(accuracy_text(yelp_model, y_val, device), 3))

## 7. Definition of done

Expected: `7 passed`, then `83 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_11.py -q

In [ ]:
!pytest -m fast -q

## 8. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 11: text as tensors"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main